In [ ]:
import dagster as dg

#from resources import MyAssetConfig

import dagster as dg

class MyAssetConfig(dg.Config):
    person_name: str

@dg.definitions
def resources() -> dg.Definitions:
    return dg.Definitions(resources={"config": MyAssetConfig(person_name="")})
    
@dg.asset
def greeting(config: MyAssetConfig) -> str:
    return f"hello {config.person_name}"


asset_result = dg.materialize(
    [greeting],
    run_config=dg.RunConfig({"greeting": MyAssetConfig(person_name="Alice")}),
)

print(asset_result)

In [ ]:
import dagster as dg

class UserData(dg.Config):
    age: int
    email: str
    profile_picture_url: str

class MyNestedConfig(dg.Config):
    user_data: dict[str, UserData]

@dg.asset
def average_age(config: MyNestedConfig): ...

result = dg.materialize(
    [average_age],
    run_config=dg.RunConfig(
        {
            "average_age": MyNestedConfig(
                user_data={
                    "Alice": UserData(
                        age=10,
                        email="alice@gmail.com",
                        profile_picture_url="...",
                    ),
                    "Bob": UserData(
                        age=20,
                        email="bob@gmail.com",
                        profile_picture_url="...",
                    ),
                }
            )
        }
    ),
)

In [ ]:
import dagster as dg
from dagster import op, job, In, Out, DynamicOut, DynamicOutput

@op(out=DynamicOut())
def node1_partition(n: int, p: int):
    """
    Für jedes m = 1 .. p-1:
    - erzeuge die ersten n Zahlen >= 2, die kongruent m mod p sind
    - yield als eigener DynamicOutput mit mapping_key = part_{m}
    """
    
    for m in range(1, p):
        part: List[int] = []

        k = 0
        while len(part) < n:
            candidate = m + k * p
            if candidate >= 2:
                part.append(candidate)
            k += 1

        yield DynamicOutput(
            value=part,
            mapping_key=f"part_{m}",
        )

@job
def test_job():
    node1_partition()

result = test_job.execute_in_process(
    run_config={"ops": {"node1_partition": {"inputs": {"n": {"value": 100}, "p": {"value": 13}}}}})

In [ ]:
import dagster as dg
from dagster import op, job, In, Out, DynamicOut, DynamicOutput, multiprocess_executor
from dataclasses import dataclass
from typing import List, Tuple, Optional
from collections import defaultdict

import math
import numpy as np

# ---------------------------
# SPECS / RESULTS
# ---------------------------

@dataclass
class Node1Spec:
    n: int
    p: int

@dataclass
class Node2Spec:
    use_numpy: bool = False

@dataclass
class Node2Result:
    value: int
    is_prime: bool

@dataclass
class Node3Spec:
    use_numpy: bool = False
    binwidth: int = 1000
    mode: str = "per_partition"

from typing import Dict

@dataclass
class Node3Result:
    counts: Dict[int, int]   

# NODE 1: Partitionierung + Fan-Out
@op(out=DynamicOut())
def node1_partition(n: int, p: int):
    """
    Für jedes m = 1 .. p-1:
    - erzeuge die ersten n Zahlen >= 2, die kongruent m mod p sind
    - yield als eigener DynamicOutput mit mapping_key = part_{m}
    """
    
    for m in range(1, p):
        part: List[int] = []

        k = 0
        while len(part) < n:
            candidate = m + k * p
            if candidate >= 2:
                part.append(candidate)
            k += 1

        yield DynamicOutput(
            value=part,
            mapping_key=f"part_{m}",
        )
        
# NODE 2: Primzahlenprüfung
@op
def node2_prime_check(partition: List[int]) -> List[Node2Result]:
    use_numpy = False
    results = []
    for num in partition:
        if num < 2:
            results.append(Node2Result(num, False))
            continue
        prime = True
        for i in range(2, int(math.isqrt(num)) + 1):
            if num % i == 0:
                prime = False
                break
        results.append(Node2Result(num, prime))
    if use_numpy:
        results = np.array(results)
    return results


# NODE 3: Statistik / Binning
@op
def node3_stat(partition_results: List[Node2Result]) -> Node3Result:

    binwidth = 10
    counts = defaultdict(int)

    for r in partition_results:
        if r.is_prime:
            idx = r.value // binwidth
            counts[idx] += 1

    return Node3Result(counts=dict(counts))


# NODE 4: Aggregation
@op
def node4_aggregate(stats_list: List[Node3Result]) -> Node3Result:
    agg = defaultdict(int)

    for stats in stats_list:
        for idx, cnt in stats.counts.items():
            agg[idx] += cnt

    return Node3Result(counts=dict(agg))


@job(executor_def=multiprocess_executor)
def test_job():
    partitions= node1_partition()

    node2_results = partitions.map(
        lambda p: node2_prime_check(p)
    )

    node3_stats = node2_results.map(
        lambda r: node3_stat(r)
    )

    node4_aggregate(node3_stats.collect())

In [ ]:
from jobs import test_job

result = test_job.execute_in_process(
    run_config={"ops": {"node1_partition": {"inputs": {"n": {"value": 100}, "p": {"value": 3}}}}})

agg_result = result.output_for_node("node4_aggregate")
print(result)

In [1]:
from jobs import test_job
from dagster import DagsterInstance, execute_job, job, reconstructable

instance = DagsterInstance.get()
run_config={"ops": {"node1_partition": {"inputs": {"n": {"value": 100000}, "p": {"value": 3}}}}}
with execute_job(reconstructable(test_job),run_config=run_config, instance=instance) as result:
    agg_result = result.output_for_node("node4_aggregate")
    
print("Result of Job: ", agg_result)    

No dagster instance configuration file (dagster.yaml) found at /opt/app/dagster_home. Defaulting to loading and storing all metadata with /opt/app/dagster_home. If this is the desired behavior, create an empty dagster.yaml file in /opt/app/dagster_home.
2026-01-17 19:20:00 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42583 - RUN_START - Started execution of run for "test_job".
2026-01-17 19:20:00 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42583 - ENGINE_EVENT - Executing steps using multiprocess executor: parent process (pid: 42583)
2026-01-17 19:20:00 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42583 - node1_partition - STEP_WORKER_STARTING - Launching subprocess for "node1_partition".
2026-01-17 19:20:00 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42869 - node1_partition - STEP_WORKER_STARTED - Executing step "node1_partition" in subprocess.
2026-01-17 19:20

10
10


2026-01-17 19:20:02 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42911 - node2_prime_check[part_1] - STEP_OUTPUT - Yielded output "result" of type "[Node2Result]". (Type check passed).
2026-01-17 19:20:02 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - node2_prime_check[part_1] - Writing file at: /opt/app/dagster_home/storage/71a70a31-c61d-4a62-90e6-7c4558235ee6/node2_prime_check[part_1]/result using PickledObjectFilesystemIOManager...
2026-01-17 19:20:02 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42915 - node2_prime_check[part_2] - STEP_OUTPUT - Yielded output "result" of type "[Node2Result]". (Type check passed).
2026-01-17 19:20:02 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - node2_prime_check[part_2] - Writing file at: /opt/app/dagster_home/storage/71a70a31-c61d-4a62-90e6-7c4558235ee6/node2_prime_check[part_2]/result using PickledObjectFilesystemIOManager...
20

15
15


2026-01-17 19:20:04 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 42583 - node4_aggregate - STEP_WORKER_STARTING - Launching subprocess for "node4_aggregate".
2026-01-17 19:20:05 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 43076 - node4_aggregate - STEP_WORKER_STARTED - Executing step "node4_aggregate" in subprocess.
2026-01-17 19:20:05 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 43076 - node4_aggregate - RESOURCE_INIT_STARTED - Starting initialization of resources [io_manager].
2026-01-17 19:20:05 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 43076 - node4_aggregate - RESOURCE_INIT_SUCCESS - Finished initialization of resources [io_manager].
2026-01-17 19:20:05 +0000 - dagster - DEBUG - test_job - 71a70a31-c61d-4a62-90e6-7c4558235ee6 - 43076 - LOGS_CAPTURED - Started capturing logs in process (pid: 43076).
2026-01-17 19:20:05 +0000 - dagster - DEBUG - test_job - 

Result of Job:  Node3Result(counts={0: 1228, 1: 1033, 2: 983, 3: 958, 4: 930, 5: 924, 6: 878, 7: 902, 8: 876, 9: 879, 10: 861, 11: 848, 12: 858, 13: 851, 14: 838, 15: 835, 16: 814, 17: 845, 18: 828, 19: 814, 20: 823, 21: 811, 22: 819, 23: 784, 24: 823, 25: 793, 26: 805, 27: 790, 28: 792, 29: 773})


/usr/local/lib/python3.11/site-packages/dagster/_core/execution/context_creation_job.py:276: RuntimeWarning: coroutine 'BaseEventLoop.shutdown_asyncgens' was never awaited
  pass
